# Bead-count sweep correction examples

Matches the SNR-sweep examples using the same plotting helpers: acquired focal positions in rows; uncorrected and three corrected states in columns; shared normalized intensity; residual pupil-phase insets fixed at ±2π; 6 × 6 inch figures, PNG and PDF exports at 300 DPI.

Reconstructs each exact saved bead geometry, including overlaps, and uses saved cumulative SLM commands and fixed exposure. Fresh seeded Poisson images illustrate each state, not the original acquired noise. The final state is simulated after loop 3. No optimization is rerun.

Companion graphs use the saved per-iteration optimizer coefficients. Curves show estimation RMS before clipping; black crosses show residual RMS after the applied correction. Both archives are read without modifying the sweep results.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while not ((ROOT / 'utils').is_dir() and (ROOT / 'optimization').is_dir()):
    if ROOT == ROOT.parent:
        raise RuntimeError('Run this notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import importlib
from utils import psf
from systematic_testing.correction_examples import plot_pd_correction_examples as correction_plots
from systematic_testing.correction_examples import plot_pd_iteration_rms as iteration_plots
importlib.reload(correction_plots)
importlib.reload(iteration_plots)
from systematic_testing.correction_examples.plot_pd_correction_examples import plot_correction_example
RESULTS_PATH = ROOT / 'systematic_testing/bead_count_sweep/results/pd_rms_bead_count_sweep_snr15_results.npz'
HISTORY_PATH = ROOT / 'systematic_testing/bead_count_sweep/results/pd_rms_bead_count_sweep_snr15_iteration_histories.npz'
OUTPUT_DIR = ROOT / 'systematic_testing/bead_count_sweep/figures/correction_examples_acquired_planes'
# (initial RMS in waves, bead count, ZERO-BASED trial).
EXAMPLE_CASES = [(0.05, 1, 0), (0.10, 1, 2), (0.10, 3, 2),
                 (0.10, 5, 2), (0.15, 1, 0), (0.15, 4, 1),
                 (0.20, 3, 0), (0.30, 5, 2)]
NOISY = True
plt.style.use('bmh')
plt.rcParams.update({'font.size': 12, 'text.usetex': True, 'figure.dpi': 300})


In [ ]:
# Opening the file once is safe across the sweep's atomic checkpoint replacements.
with RESULTS_PATH.open('rb') as stream:
    with np.load(stream, allow_pickle=False) as archive:
        saved = {key: archive[key].copy() for key in archive.files}
complete_runs = saved['completed'].all(axis=-1)
print(f'Completed trajectories: {complete_runs.sum()}/{complete_runs.size}')
print('Acquired focal positions [um]:', saved['focal_positions'] * 1000)
print('Calibration SNR:', float(saved['target_snr']))
print('Available (RMS, beads, zero-based trial):')
for ri, ci, trial in np.argwhere(complete_runs):
    print((float(saved['initial_rms'][ri]), int(saved['bead_counts'][ci]), int(trial)))


In [ ]:
def selected_case(rms, bead_count, trial):
    ri = np.flatnonzero(np.isclose(saved['initial_rms'], rms))
    ci = np.flatnonzero(saved['bead_counts'] == bead_count)
    if len(ri) != 1 or len(ci) != 1 or not 0 <= trial < complete_runs.shape[2]:
        raise ValueError(f'Invalid case {(rms, bead_count, trial)}')
    ri, ci = int(ri[0]), int(ci[0])
    if not complete_runs[ri, ci, trial]:
        print(f'Skipping incomplete case {(rms, bead_count, trial)}')
        return None
    return ri, ci

# Adapter preserves the shared plotting routine's array indexing.
# Its second axis selects bead count here; the title is supplied explicitly.
plot_data = dict(saved, target_snr=saved['bead_counts'])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for rms, bead_count, trial in EXAMPLE_CASES:
    indices = selected_case(rms, bead_count, trial)
    if indices is None:
        continue
    ri, ci = indices
    centers = saved['bead_positions_mm'][ci, trial, :bead_count]
    if not np.isfinite(centers).all():
        raise ValueError('Saved bead geometry is incomplete.')
    grid = psf.Centered_Square_Grid(L_ffp=0.0065, grid_ffp=32, z_level=0)
    sample = psf.bead_img_3D(grid, saved['sample_z'],
        centers[:, 0], centers[:, 1], centers[:, 2],
        np.full(bead_count, float(saved['bead_radius_mm'])))
    title = (f'Initial RMS {rms:g} waves | {bead_count} beads | '
             f'calibration SNR {float(saved["target_snr"]):g} | trial {trial+1}')
    fig, axes = plot_correction_example(plot_data, rms, bead_count, trial,
        noisy=NOISY, acquired_planes=True, sample=sample, example_title=title)
    stem = f'rms{rms:g}_beads{bead_count}_trial{trial+1}'
    stem += '_noisy' if NOISY else '_clean'
    for extension in ('png', 'pdf'):
        fig.savefig(OUTPUT_DIR / f'{stem}_acquired_planes.{extension}', dpi=300)
    plt.show()
    plt.close(fig)


## Correction RMS versus optimizer iterations

Matches the SNR examples in size, fonts, colors, legend and loop-boundary annotations. Each loop has a separate estimation-error curve; black crosses show the saved applied residual after clipping. The cumulative iteration axis excludes random-start screening evaluations. No optimization is rerun.


In [ ]:
with np.load(HISTORY_PATH, allow_pickle=False) as histories:
    # Ensure the histories belong to this checkpoint before plotting.
    for key in ('initial_rms', 'bead_counts', 'modes', 'optimizer_steps',
                'truth_strengths', 'slm_strengths', 'residual_rms'):
        if not np.array_equal(histories[key], saved[key]):
            raise ValueError(f'History/checkpoint mismatch: {key}')
    for rms, bead_count, trial in EXAMPLE_CASES:
        if selected_case(rms, bead_count, trial) is None:
            continue
        fig, ax = iteration_plots.plot_iteration_rms(
            plot_data, rms, bead_count, trial, histories=histories)
        stem = f'rms{rms:g}_beads{bead_count}_trial{trial+1}_iteration_rms'
        for extension in ('png', 'pdf'):
            fig.savefig(OUTPUT_DIR / f'{stem}.{extension}', dpi=300)
        plt.show()
        plt.close(fig)


## Generated example pairs

### RMS 0.05 waves, 1 beads, trial 1

![Correction images](figures/correction_examples_acquired_planes/rms0.05_beads1_trial1_noisy_acquired_planes.png)

![Optimizer RMS](figures/correction_examples_acquired_planes/rms0.05_beads1_trial1_iteration_rms.png)

### RMS 0.1 waves, 1 beads, trial 3

![Correction images](figures/correction_examples_acquired_planes/rms0.1_beads1_trial3_noisy_acquired_planes.png)

![Optimizer RMS](figures/correction_examples_acquired_planes/rms0.1_beads1_trial3_iteration_rms.png)

### RMS 0.1 waves, 3 beads, trial 3

![Correction images](figures/correction_examples_acquired_planes/rms0.1_beads3_trial3_noisy_acquired_planes.png)

![Optimizer RMS](figures/correction_examples_acquired_planes/rms0.1_beads3_trial3_iteration_rms.png)

### RMS 0.1 waves, 5 beads, trial 3

![Correction images](figures/correction_examples_acquired_planes/rms0.1_beads5_trial3_noisy_acquired_planes.png)

![Optimizer RMS](figures/correction_examples_acquired_planes/rms0.1_beads5_trial3_iteration_rms.png)

### RMS 0.15 waves, 1 beads, trial 1

![Correction images](figures/correction_examples_acquired_planes/rms0.15_beads1_trial1_noisy_acquired_planes.png)

![Optimizer RMS](figures/correction_examples_acquired_planes/rms0.15_beads1_trial1_iteration_rms.png)

### RMS 0.15 waves, 4 beads, trial 2

![Correction images](figures/correction_examples_acquired_planes/rms0.15_beads4_trial2_noisy_acquired_planes.png)

![Optimizer RMS](figures/correction_examples_acquired_planes/rms0.15_beads4_trial2_iteration_rms.png)

### RMS 0.2 waves, 3 beads, trial 1

![Correction images](figures/correction_examples_acquired_planes/rms0.2_beads3_trial1_noisy_acquired_planes.png)

![Optimizer RMS](figures/correction_examples_acquired_planes/rms0.2_beads3_trial1_iteration_rms.png)

### RMS 0.3 waves, 5 beads, trial 3

![Correction images](figures/correction_examples_acquired_planes/rms0.3_beads5_trial3_noisy_acquired_planes.png)

![Optimizer RMS](figures/correction_examples_acquired_planes/rms0.3_beads5_trial3_iteration_rms.png)

## Percentage of AO loops that helped

A loop helped when its saved applied residual RMS (after clipping) was strictly lower than immediately before that loop. Each case pools three trials × three loops (nine loops); calibration SNR is 15. This measures improvement, not attainment of a correction threshold.

Overall: **232/270 loops (85.9%)** helped. Loop 1: 80.0%; loops 2 and 3: 88.9% each.

| Initial RMS [waves] | 1 bead | 2 beads | 3 beads | 4 beads | 5 beads |
|---|---:|---:|---:|---:|---:|
| 0.05 | 88.9% (8/9) | 100.0% (9/9) | 100.0% (9/9) | 88.9% (8/9) | 100.0% (9/9) |
| 0.1 | 66.7% (6/9) | 77.8% (7/9) | 100.0% (9/9) | 77.8% (7/9) | 77.8% (7/9) |
| 0.15 | 77.8% (7/9) | 100.0% (9/9) | 55.6% (5/9) | 55.6% (5/9) | 77.8% (7/9) |
| 0.2 | 100.0% (9/9) | 88.9% (8/9) | 88.9% (8/9) | 100.0% (9/9) | 100.0% (9/9) |
| 0.25 | 88.9% (8/9) | 77.8% (7/9) | 100.0% (9/9) | 100.0% (9/9) | 88.9% (8/9) |
| 0.3 | 88.9% (8/9) | 77.8% (7/9) | 88.9% (8/9) | 66.7% (6/9) | 77.8% (7/9) |

Detailed per-loop percentages: [CSV](results/loop_improvement_summary.csv).
